In [1]:
import tensorflow as tf
import pubchempy as pcp
import numpy as np
import pickle
from pathlib import Path

# Load TensorFlow model and LabelEncoder
BASE_DIR = Path(__file__).resolve().parent.parent.parent
model = tf.keras.models.load_model(BASE_DIR / 'models/best_model.h5')
with open(BASE_DIR / 'models/label_encoder.pkl', 'rb') as f:
    LE = pickle.load(f)

def predict_interaction(drug_cid: int, excipient_cid: int):
    drug = pcp.Compound.from_cid(drug_cid)
    excipient = pcp.Compound.from_cid(excipient_cid)
    fpd = list(drug.cactvs_fingerprint)
    fpe = list(excipient.cactvs_fingerprint)
    combined = fpd + fpe

    X_predict = np.array(combined).reshape(1, -1).astype(np.int64)
    if X_predict.shape[1] != 1762:
        raise ValueError("Input fingerprint length must be 1762")

    prediction = model.predict(X_predict, verbose=0)
    rounded = np.round(prediction).astype(int)
    result = LE.inverse_transform(rounded)[0]
    compatibility = "compatible" if result == 1 else "incompatible"
    score = float(prediction[0][0]) * 100 if result == 1 else (1 - float(prediction[0][0])) * 100

    return {
        "drugCID": str(drug_cid),
        "drugName": drug.iupac_name or (drug.synonyms[0] if drug.synonyms else "Unknown"),
        "excipientCID": str(excipient_cid),
        "excipientName": excipient.iupac_name or (excipient.synonyms[0] if excipient.synonyms else "Unknown"),
        "compatibility": compatibility,
        "score": score,
        "prediction": int(rounded[0]),
        "confidence": float(prediction[0][0]),
        "notes": f"TensorFlow model prediction: {compatibility.capitalize()}",
        "fingerprint": "Generated via PubChem CACTVS",
    }

ModuleNotFoundError: No module named 'tensorflow'